# Rank vs Log Transformation: Comparing Income Prediction Models
## Under domain adaptation with no target labels

Both rank and log are **monotone transformations** that stabilise variance
and compress outliers before fitting a regression. But they behave very
differently when the target income range lies outside the training range.

| Property | Rank transform | Log transform |
|---|---|---|
| Output space | [0, 1] — bounded | (0, ∞) — unbounded |
| Extrapolation | Rescaled to estimated range at inference | Exponential extrapolation |
| Handles out-of-range targets | Via range rescaling step | Directly (if log-linear holds) |
| Sensitive to range estimate | Yes — rescaling depends on it | No — extrapolates naturally |
| Assumes income distribution | Uniform (rank) | Log-normal |
| Interpretability | Percentile → dollar | Log-dollar → dollar |

**Key question:** Does the log model extrapolate better than the rank model
for stores whose income range is outside the training range?

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import rankdata, spearmanr, ks_2samp
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

STORES  = ["whole_foods","kroger","safeway","walmart","thrift_store"]
LABELS  = ["Whole Foods","Kroger","Safeway","Walmart","Thrift"]
TIERS   = ["High","Median","Median","Median","Low"]
COLORS  = ["#185FA5","#1D9E75","#0F6E56","#854F0B","#A32D2D"]

FEAT_COLS = ["age","visits_per_month","avg_basket_usd","monthly_spend_usd",
             "grocery_pct","electronics_pct","apparel_pct","home_pct",
             "private_label_pct","online_orders_pct","coupon_usage_pct",
             "loyalty_score","segment_enc"]
FEAT_NICE = ["Age","Visits/mo","Basket $","Monthly $","Grocery %",
             "Elec %","Apparel %","Home %","Priv.label","Online %",
             "Coupon %","Loyalty","Segment"]
PROXY_IDX   = [2, 9]
BEST_ALPHAS = [0.1, 0.3, 0.5, 0.7]

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5, "font.size": 10,
})
print("Setup complete.")

## 2. Load data

In [ ]:
df = pd.read_csv("grocery_all_stores.csv")
le = LabelEncoder(); le.fit(df["segment"]); df["segment_enc"] = le.transform(df["segment"])

TRUE_RANGES = {"whole_foods":(85000,160000),"kroger":(55000,105000),
               "safeway":(45000,95000),"walmart":(30000,80000),"thrift_store":(10000,45000)}

sc_all    = StandardScaler()
X_all_sc  = sc_all.fit_transform(df[FEAT_COLS].values.astype(float))
pca_ref   = PCA(n_components=6, random_state=42)
X_all_pca = pca_ref.fit_transform(X_all_sc)
ref_lib   = {s: {"centroid": X_all_pca[df["store"]==s].mean(axis=0),
                 "income_mean": np.mean(TRUE_RANGES[s])} for s in STORES}

def make_bridge(Xs, Xt, yr, alphas=BEST_ALPHAS, n=400, seed=42):
    rng = np.random.RandomState(seed); bX, by = [], []
    for a in alphas:
        for _ in range(n // len(alphas)):
            i, j = rng.randint(len(Xs)), rng.randint(len(Xt))
            bX.append((1-a)*Xs[i] + a*Xt[j])
            by.append((1-a)*yr[i]  + a*0.5)
    return np.array(bX), np.array(by)

def get_range(Xs, Xt, ys, ts, refs):
    preg = LinearRegression().fit(Xs[:,PROXY_IDX], ys)
    gap  = preg.predict(Xt[:,PROXY_IDX]).mean() - preg.predict(Xs[:,PROXY_IDX]).mean()
    cen  = X_all_pca[df["store"]==ts].mean(axis=0).reshape(1,-1)
    dists = {s: cdist(cen, ref_lib[s]["centroid"].reshape(1,-1),"euclidean")[0,0] for s in refs}
    nn    = min(dists, key=dists.get)
    bl    = 0.4*ref_lib[nn]["income_mean"] + 0.6*(preg.predict(Xs[:,PROXY_IDX]).mean()+gap)
    span  = (ys.max()-ys.min()) / 2 * 1.1
    return max(0, round((bl-span)/1000)*1000), round((bl+span)/1000)*1000

print(f"Loaded {len(df)} rows across {df['store'].nunique()} stores.")
for s, lbl in zip(STORES, LABELS):
    sub = df[df["store"]==s]
    print(f"  {lbl:<15}  n={len(sub)}  "
          f"income=${sub['income_usd'].min():,.0f}–${sub['income_usd'].max():,.0f}")

## 3. Why log? — visualising income distributions and log-normality

Income distributions are typically right-skewed.
After a log transform, the distribution becomes much closer to normal (Gaussian).
This is good for linear regression: it stabilises variance and reduces
the leverage of extreme high-income outliers.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("Income distributions: raw vs log-transformed", fontweight="bold")

for i, (ts, lbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    y = df[df["store"]==ts]["income_usd"].values

    # Row 0: raw income
    ax = axes[0, i]
    ax.hist(y/1000, bins=25, color=col, alpha=0.75, edgecolor="white", density=True)
    ax.set_title(f"{lbl}\nraw income", fontweight="bold", color=col, fontsize=9)
    ax.set_xlabel("Income ($k)"); ax.set_ylabel("Density" if i==0 else "")
    # Add normal fit
    from scipy.stats import norm
    mu, sig = (y/1000).mean(), (y/1000).std()
    xs = np.linspace(mu-3*sig, mu+3*sig, 200)
    ax.plot(xs, norm.pdf(xs, mu, sig), "k--", linewidth=1.5, alpha=0.6)

    # Row 1: log income
    ax = axes[1, i]
    log_y = np.log(y)
    ax.hist(log_y, bins=25, color=col, alpha=0.75, edgecolor="white", density=True)
    ax.set_title("log(income)", fontweight="bold", color=col, fontsize=9)
    ax.set_xlabel("log(income $)"); ax.set_ylabel("Density" if i==0 else "")
    mu_l, sig_l = log_y.mean(), log_y.std()
    xs_l = np.linspace(mu_l-3*sig_l, mu_l+3*sig_l, 200)
    ax.plot(xs_l, norm.pdf(xs_l, mu_l, sig_l), "k--", linewidth=1.5, alpha=0.6,
            label="Normal fit")

axes[1, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Skewness table
print("Skewness comparison (closer to 0 = more normal):")
print(f"  {'Store':<15}  {'Raw skew':>10}  {'Log skew':>10}")
print("  " + "-"*38)
for ts, lbl in zip(STORES, LABELS):
    y = df[df["store"]==ts]["income_usd"].values
    from scipy.stats import skew
    print(f"  {lbl:<15}  {skew(y):>10.3f}  {skew(np.log(y)):>10.3f}")

## 4. The two model architectures

### Rank model (S10)
```
y_rank    = (rankdata(y_source) - 1) / (N - 1)    # maps income to [0, 1]
model.fit(X_source + bridges, y_rank)
pred_rank = clip(model.predict(X_target), 0, 1)
pred_$    = lo + pred_rank * (hi - lo)             # rescale to estimated range
```

### Log model (new)
```
y_log     = log(y_source)                          # maps income to log space
model.fit(X_source + bridges_log, y_log)
pred_log  = model.predict(X_target)
pred_$    = exp(pred_log)                          # back to dollar space
```

**Key difference:** The log model extrapolates naturally in log space.
The rank model always needs a range estimate for the rescaling step.
If the range estimate is accurate, rank wins.
If the log-linear relationship holds beyond the training range, log wins.

In [ ]:
def make_bridge_log(Xs, Xt, y_log, alphas=BEST_ALPHAS, n=400, seed=42):
    """Bridge samples for log model — labels are log-income values."""
    rng = np.random.RandomState(seed); bX, by = [], []
    # For log bridges, the target label prior is log(median_source_income)
    log_prior = np.median(y_log)
    for a in alphas:
        for _ in range(n // len(alphas)):
            i, j = rng.randint(len(Xs)), rng.randint(len(Xt))
            bX.append((1-a)*Xs[i] + a*Xt[j])
            # Label: interpolate toward log-space median (= source median as prior)
            by.append((1-a)*y_log[i] + a*log_prior)
    return np.array(bX), np.array(by)

def run_rank_model(Xs, Xt, ys, lo, hi):
    yr = (rankdata(ys)-1)/(len(ys)-1)
    bX, by = make_bridge(Xs, Xt, yr)
    m = Ridge(alpha=10.0)
    m.fit(np.vstack([Xs,bX]), np.concatenate([yr,by]))
    preds = lo + np.clip(m.predict(Xt), 0, 1)*(hi-lo)
    return preds, m

def run_log_model(Xs, Xt, ys):
    y_log = np.log(ys)
    bX, by = make_bridge_log(Xs, Xt, y_log)
    m = Ridge(alpha=10.0)
    m.fit(np.vstack([Xs,bX]), np.concatenate([y_log,by]))
    preds = np.exp(m.predict(Xt))
    return preds, m

print("Model functions defined.")
print("Rank model: normalises income to [0,1] rank, rescales at inference.")
print("Log  model: normalises income to log space, exponentiates at inference.")

## 5. Run leave-one-store-out CV for both models

In [ ]:
results = {}

for ts in STORES:
    src = df[df["store"]!=ts].copy()
    tgt = df[df["store"]==ts].copy()
    sc  = StandardScaler()
    Xs  = sc.fit_transform(src[FEAT_COLS].values.astype(float))
    Xt  = sc.transform(tgt[FEAT_COLS].values.astype(float))
    ys  = src["income_usd"].values
    yt  = tgt["income_usd"].values
    refs = [s for s in STORES if s!=ts]
    lo, hi = get_range(Xs, Xt, ys, ts, refs)

    # Rank model
    pred_rank, m_rank = run_rank_model(Xs, Xt, ys, lo, hi)

    # Log model
    pred_log, m_log = run_log_model(Xs, Xt, ys)

    sp_rank, _ = spearmanr(yt, pred_rank)
    sp_log,  _ = spearmanr(yt, pred_log)

    results[ts] = {
        "yt": yt, "Xs": Xs, "Xt": Xt, "ys": ys,
        "lo": lo, "hi": hi,
        "rank": {
            "preds":    pred_rank,
            "mae":      mean_absolute_error(yt, pred_rank),
            "r2":       r2_score(yt, pred_rank),
            "spearman": sp_rank,
            "bias":     pred_rank.mean() - yt.mean(),
        },
        "log": {
            "preds":    pred_log,
            "mae":      mean_absolute_error(yt, pred_log),
            "r2":       r2_score(yt, pred_log),
            "spearman": sp_log,
            "bias":     pred_log.mean() - yt.mean(),
        },
    }
    lbl = LABELS[STORES.index(ts)]
    print(f"  {lbl:<15}  "
          f"RANK: MAE=${results[ts]['rank']['mae']:>8,.0f}  R²={results[ts]['rank']['r2']:>7.4f}  sp={sp_rank:.4f}  |  "
          f"LOG:  MAE=${results[ts]['log']['mae']:>8,.0f}  R²={results[ts]['log']['r2']:>7.4f}  sp={sp_log:.4f}")

## 6. Head-to-head comparison: MAE, R², Spearman, Bias

In [ ]:
metrics = [("MAE ($)", "mae", lambda v: v/1000, "lower better"),
           ("R²",      "r2",  lambda v: v,       "higher better"),
           ("Spearman r","spearman",lambda v: v,  "higher better"),
           ("Bias ($)", "bias",lambda v: v/1000,  "closer to 0")]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Rank vs Log model: head-to-head on all four metrics", fontweight="bold")

for ax, (title, key, tfm, direction) in zip(axes.flat, metrics):
    x   = np.arange(len(STORES))
    rv  = [tfm(results[s]["rank"][key]) for s in STORES]
    lv  = [tfm(results[s]["log"][key])  for s in STORES]
    ax.bar(x-0.2, rv, 0.35, label="Rank model", color=[c+"99" for c in COLORS],
           edgecolor=COLORS, linewidth=1.5)
    ax.bar(x+0.2, lv, 0.35, label="Log model",  color=COLORS,
           edgecolor="white", alpha=0.85, hatch="//")
    ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(f"{title}  ({direction})", fontweight="bold")
    ax.legend(fontsize=8)

    # Annotate winner for each store
    for i, (r, l) in enumerate(zip(rv, lv)):
        if key == "mae" or key == "bias":
            rank_wins = abs(r) < abs(l)
        else:
            rank_wins = r > l
        winner = "R" if rank_wins else "L"
        col    = COLORS[i]
        ypos   = max(abs(r), abs(l)) * (1.05 if key!="bias" else 1.15)
        ax.text(i, ypos if key in ["mae","spearman","r2"] else min(r,l)-0.3,
                winner, ha="center", fontsize=9, fontweight="bold", color=col)

plt.tight_layout(); plt.show()

# Summary table
print("\nSummary (R = rank wins, L = log wins):")
print(f"  {'Store':<15}  {'RANK MAE':>10}  {'LOG MAE':>10}  {'Winner':>8}  "
      f"{'RANK R²':>9}  {'LOG R²':>9}  {'Winner':>8}")
print("  "+"-"*80)
for ts, lbl in zip(STORES, LABELS):
    rm = results[ts]["rank"]["mae"]/1000
    lm = results[ts]["log"]["mae"]/1000
    rr = results[ts]["rank"]["r2"]
    lr = results[ts]["log"]["r2"]
    mw = "RANK" if rm < lm else "LOG "
    rw = "RANK" if rr > lr else "LOG "
    print(f"  {lbl:<15}  ${rm:>9.2f}k  ${lm:>9.2f}k  {mw:>8}  "
          f"{rr:>9.4f}  {lr:>9.4f}  {rw:>8}")

## 7. Actual vs predicted scatter — where does each model struggle?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 9))
fig.suptitle("Actual vs predicted: Rank (top) vs Log (bottom)", fontweight="bold")

for i, (ts, lbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    r   = results[ts]
    yt  = r["yt"]
    lo  = TRUE_RANGES[ts][0]; hi = TRUE_RANGES[ts][1]

    for row, (model_key, model_lbl) in enumerate([("rank","Rank"), ("log","Log")]):
        ax    = axes[row, i]
        preds = r[model_key]["preds"]
        mae   = r[model_key]["mae"]
        rr    = r[model_key]["r2"]
        sp    = r[model_key]["spearman"]

        ax.scatter(yt/1000, preds/1000, c=col, s=10, alpha=0.45, edgecolors="none")
        ax.plot([lo/1000,hi/1000],[lo/1000,hi/1000],"k--",linewidth=1,alpha=0.5)
        ax.set_xlabel("Actual ($k)"); ax.set_ylabel("Predicted ($k)")
        ax.set_title(
            f"{lbl} — {model_lbl}\n"
            f"MAE=${mae/1000:.1f}k  R²={rr:.3f}  sp={sp:.3f}",
            fontsize=8.5, fontweight="bold", color=col)
        # Add residual colour strip: green=underpredict, red=overpredict
        for j in range(len(yt)):
            signed = preds[j] - yt[j]
            ax.scatter(yt[j]/1000, preds[j]/1000,
                       c="#791F1F" if signed>0 else "#27500A",
                       s=8, alpha=0.3, edgecolors="none")

plt.tight_layout(); plt.show()

## 8. Residual analysis — distribution and patterns

A good model has residuals (actual − predicted) that are:
- Centred near zero (low bias)
- Symmetric (no skew toward over or under)
- Homoscedastic (same spread across all income levels)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("Residuals: Rank (top) vs Log (bottom)  [actual - predicted]",
             fontweight="bold")

for i, (ts, lbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    yt = results[ts]["yt"]
    for row, model_key in enumerate(["rank","log"]):
        ax    = axes[row, i]
        preds = results[ts][model_key]["preds"]
        resid = (yt - preds) / 1000   # actual - pred in $k

        ax.scatter(preds/1000, resid, c=col, s=10, alpha=0.4, edgecolors="none")
        ax.axhline(0, color="black", linewidth=1)
        ax.axhline(resid.mean(), color=col, linewidth=2, linestyle="--",
                   label=f"mean={resid.mean():+.1f}k")
        # ±1 std band
        ax.axhline(resid.mean()+resid.std(), color=col, linewidth=0.8,
                   linestyle=":", alpha=0.6)
        ax.axhline(resid.mean()-resid.std(), color=col, linewidth=0.8,
                   linestyle=":", alpha=0.6, label=f"std={resid.std():.1f}k")

        ax.set_xlabel("Predicted ($k)"); ax.set_ylabel("Residual ($k)")
        ax.set_title(f"{lbl}\n{'Rank' if row==0 else 'Log'} model",
                     fontsize=9, fontweight="bold", color=col)
        ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

# Residual stats table
print("Residual statistics (actual - predicted):")
print(f"  {'Store':<15}  {'Model':>6}  {'Mean ($k)':>10}  {'Std ($k)':>10}  {'Skew':>7}")
print("  "+"-"*52)
from scipy.stats import skew
for ts, lbl in zip(STORES, LABELS):
    yt = results[ts]["yt"]
    for mk in ["rank","log"]:
        resid = (yt - results[ts][mk]["preds"]) / 1000
        print(f"  {lbl:<15}  {mk:>6}  {resid.mean():>+10.2f}  "
              f"{resid.std():>10.2f}  {skew(resid):>7.3f}")

## 9. Extrapolation: what happens below and above the training range?

The crucial test: the training data covers $30k–$105k.
How do rank and log models behave when we ask them to predict
income at $10k (below training floor) and $150k (above training ceiling)?

For the rank model, extrapolation is handled by the rescaling step.
For the log model, extrapolation is handled by Ridge in log space.
This cell isolates the extrapolation behaviour by using a fixed
narrow source (Kroger only) and a wide target range.

In [ ]:
# Use Kroger as the only source (narrow range $55k-$105k)
# Predict for WF ($85k-$160k) and Thrift ($10k-$45k)
src_narrow = df[df["store"]=="kroger"].copy()
sc_n = StandardScaler()
Xs_n = sc_n.fit_transform(src_narrow[FEAT_COLS].values.astype(float))
ys_n = src_narrow["income_usd"].values

# Synthetic income grid — predict across the full $5k-$175k range
x_mean = Xs_n.mean(axis=0)   # average Kroger customer feature vector
# Create customers who vary only in avg_basket_usd (the best income proxy)
basket_range = np.linspace(10, 400, 200)   # $10 to $400 basket size
X_grid = np.tile(x_mean, (200,1))
# Standardise basket index: index 2 in FEAT_COLS
basket_std_vals = (basket_range - src_narrow["avg_basket_usd"].mean()) / src_narrow["avg_basket_usd"].std()
X_grid[:,2] = basket_std_vals

# Also need to make bridge samples — use Kroger customers as both source and target
# (the extrapolation test is about what the model predicts outside training range)
lo_wf,  hi_wf  = 90000, 150000
lo_th,  hi_th  = 10000,  50000

# Rank model predictions
yr_n = (rankdata(ys_n)-1)/(len(ys_n)-1)
bX_g, by_g = make_bridge(Xs_n, X_grid, yr_n, n=200)
m_rank_n = Ridge(alpha=10.0)
m_rank_n.fit(np.vstack([Xs_n,bX_g]), np.concatenate([yr_n,by_g]))

rank_pred_wf  = lo_wf  + np.clip(m_rank_n.predict(X_grid),0,1)*(hi_wf-lo_wf)
rank_pred_th  = lo_th  + np.clip(m_rank_n.predict(X_grid),0,1)*(hi_th-lo_th)
rank_pred_src = ys_n.min() + np.clip(m_rank_n.predict(X_grid),0,1)*(ys_n.max()-ys_n.min())

# Log model predictions
y_log_n = np.log(ys_n)
log_prior = np.median(y_log_n)
bX_lg, by_lg = [], []
rng = np.random.RandomState(42)
for a in BEST_ALPHAS:
    for _ in range(50):
        i,j = rng.randint(len(Xs_n)), rng.randint(len(X_grid))
        bX_lg.append((1-a)*Xs_n[i]+a*X_grid[j])
        by_lg.append((1-a)*y_log_n[i]+a*log_prior)
m_log_n = Ridge(alpha=10.0)
m_log_n.fit(np.vstack([Xs_n,bX_lg]), np.concatenate([y_log_n,by_lg]))
log_pred = np.exp(m_log_n.predict(X_grid))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Extrapolation behaviour: Rank vs Log (source = Kroger only)",
             fontweight="bold")

# Left: rank model with different rescalings
ax = axes[0]
ax.plot(basket_range, rank_pred_src/1000, color="#1D9E75", linewidth=2,
        label="Rank: rescaled to source range ($55k-$105k)")
ax.plot(basket_range, rank_pred_wf/1000, color="#185FA5", linewidth=2, linestyle="--",
        label="Rank: rescaled to WF range ($90k-$150k)")
ax.plot(basket_range, rank_pred_th/1000, color="#A32D2D", linewidth=2, linestyle=":",
        label="Rank: rescaled to Thrift range ($10k-$50k)")
ax.axvspan(src_narrow["avg_basket_usd"].min(), src_narrow["avg_basket_usd"].max(),
           alpha=0.08, color="#1D9E75", label="Kroger basket range")
ax.set_xlabel("Avg basket size ($)"); ax.set_ylabel("Predicted income ($k)")
ax.set_title("Rank model: extrapolation by rescaling", fontweight="bold")
ax.legend(fontsize=8)

# Right: log model (no rescaling needed)
ax = axes[1]
ax.plot(basket_range, log_pred/1000, color="#185FA5", linewidth=2.5,
        label="Log model: direct prediction")
ax.plot(basket_range, rank_pred_src/1000, color="#1D9E75", linewidth=2,
        linestyle="--", label="Rank (source range) for reference")
ax.axvspan(src_narrow["avg_basket_usd"].min(), src_narrow["avg_basket_usd"].max(),
           alpha=0.08, color="#1D9E75", label="Kroger basket range")
ax.axhline(np.log(ys_n.mean()), color="gray", linewidth=1, linestyle=":",
           alpha=0.5)
ax.set_xlabel("Avg basket size ($)"); ax.set_ylabel("Predicted income ($k)")
ax.set_title("Log model: natural extrapolation in log space", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Observation:")
print("  Rank model: shape is fixed, only the y-axis scale changes with the range estimate.")
print("  Log model:  shape is fixed in log space, dollar predictions extend naturally beyond training range.")
print("  Both models predict the same RANK ordering — they differ only in the dollar scale.")

## 10. When does each model win?

The log model wins when:
1. The log-linear assumption holds across domains (richer customers really do
   spend proportionally more, not additionally more)
2. The range estimate in the rank model is badly wrong
3. The target income range spans many orders of magnitude

The rank model wins when:
1. The range estimate is accurate
2. The income-feature relationship is not log-linear across domains
3. The training and target income ranges have substantial overlap

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("When does each model win? — diagnostic plots", fontweight="bold")

# Panel 1: MAE difference vs KS shift
ax = axes[0,0]
ks_vals    = [np.mean([ks_2samp(results[ts]["Xs"][:,i],
                                 results[ts]["Xt"][:,i]).statistic
                        for i in range(len(FEAT_COLS))]) for ts in STORES]
mae_diff   = [(results[ts]["rank"]["mae"] - results[ts]["log"]["mae"])/1000
              for ts in STORES]   # positive = log wins
for ts, lbl, col, ks, md in zip(STORES, LABELS, COLORS, ks_vals, mae_diff):
    ax.scatter(ks, md, c=col, s=120, zorder=5, edgecolors="white", linewidths=1)
    ax.annotate(lbl, (ks, md), textcoords="offset points", xytext=(5,3),
                fontsize=8.5, color=col)
ax.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.6,
           label="Break-even")
ax.set_xlabel("Avg KS shift (feature shift magnitude)")
ax.set_ylabel("MAE(Rank) - MAE(Log) ($k) [positive = Log model wins]")
ax.set_title("Feature shift vs which model wins", fontweight="bold")
ax.legend(fontsize=8)

# Panel 2: MAE difference vs range estimation error
ax = axes[0,1]
range_errs = [abs((results[ts]["lo"]+results[ts]["hi"])/2 -
                  (TRUE_RANGES[ts][0]+TRUE_RANGES[ts][1])/2)/1000 for ts in STORES]
for ts, lbl, col, re, md in zip(STORES, LABELS, COLORS, range_errs, mae_diff):
    ax.scatter(re, md, c=col, s=120, zorder=5, edgecolors="white", linewidths=1)
    ax.annotate(lbl, (re, md), textcoords="offset points", xytext=(5,3),
                fontsize=8.5, color=col)
ax.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.6,
           label="Break-even")
ax.set_xlabel("Range estimate error ($k)")
ax.set_ylabel("MAE(Rank) - MAE(Log) ($k)")
ax.set_title("Range error vs which model wins", fontweight="bold")
ax.legend(fontsize=8)

# Panel 3: Bias comparison
ax = axes[0,2]
x = np.arange(len(STORES)); w = 0.3
rank_bias = [results[s]["rank"]["bias"]/1000 for s in STORES]
log_bias  = [results[s]["log"]["bias"]/1000  for s in STORES]
ax.bar(x-w/2, rank_bias, w, label="Rank bias", color=[c+"88" for c in COLORS],
       edgecolor=COLORS, linewidth=1.5)
ax.bar(x+w/2, log_bias,  w, label="Log bias",  color=COLORS, edgecolor="white",
       alpha=0.85, hatch="//")
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Bias ($k)  [positive = over-predict]")
ax.set_title("Systematic bias comparison", fontweight="bold")
ax.legend(fontsize=8)

# Panel 4-6: Spearman, R², and MAE side-by-side
metrics_3 = [("Spearman r","spearman",True), ("R²","r2",True), ("MAE ($k)","mae",False)]
for ax, (mname, mkey, higher_better) in zip(axes[1,:], metrics_3):
    x  = np.arange(len(STORES)); w = 0.3
    rv = [results[s]["rank"][mkey]/(1000 if mkey=="mae" else 1) for s in STORES]
    lv = [results[s]["log"][mkey]/(1000 if mkey=="mae" else 1)  for s in STORES]
    ax.bar(x-w/2, rv, w, label="Rank", color=[c+"88" for c in COLORS],
           edgecolor=COLORS, linewidth=1.5)
    ax.bar(x+w/2, lv, w, label="Log",  color=COLORS, edgecolor="white", alpha=0.85, hatch="//")
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
    ax.set_title(f"{mname}", fontweight="bold")
    ax.legend(fontsize=8)
    # Mark winners
    for i, (r, l) in enumerate(zip(rv, lv)):
        wins_rank = r > l if higher_better else r < l
        ax.text(i, max(abs(r),abs(l))*(1.05 if mname!="Bias" else 1.2),
                "R" if wins_rank else "L",
                ha="center", fontsize=9, fontweight="bold", color=COLORS[i])

plt.tight_layout(); plt.show()

## 11. Hybrid: combine rank and log predictions

Since rank and log make different types of errors, blending their predictions
may outperform either alone.

```
pred_hybrid = w * pred_rank + (1-w) * pred_log
```

We tune w using source leave-one-out CV — no target labels used.

In [ ]:
# Tune blend weight on source LOSO
print("Tuning blend weight w (rank fraction) on source LOSO-CV...")
best_w, best_err = 0.5, np.inf
for w_cand in np.arange(0, 1.05, 0.05):
    cv_errs = []
    for ts in STORES:
        hybrid = (w_cand * results[ts]["rank"]["preds"] +
                  (1-w_cand) * results[ts]["log"]["preds"])
        cv_errs.append(mean_absolute_error(results[ts]["yt"], hybrid))
    mean_err = np.mean(cv_errs)
    if mean_err < best_err:
        best_err, best_w = mean_err, w_cand

print(f"Best blend weight: w={best_w:.2f} (rank fraction)  avg MAE=${best_err:,.0f}")

# Final hybrid predictions
hybrid_results = {}
for ts in STORES:
    h = best_w * results[ts]["rank"]["preds"] + (1-best_w) * results[ts]["log"]["preds"]
    yt = results[ts]["yt"]
    sp, _ = spearmanr(yt, h)
    hybrid_results[ts] = {
        "preds":    h,
        "mae":      mean_absolute_error(yt, h),
        "r2":       r2_score(yt, h),
        "spearman": sp,
        "bias":     h.mean() - yt.mean(),
    }

# ── Blend weight curve ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
ws = np.arange(0, 1.05, 0.05)
avg_maes = []
for w_c in ws:
    maes = [mean_absolute_error(results[ts]["yt"],
                                 w_c*results[ts]["rank"]["preds"] +
                                 (1-w_c)*results[ts]["log"]["preds"])
            for ts in STORES]
    avg_maes.append(np.mean(maes)/1000)
ax.plot(ws, avg_maes, linewidth=2.5, color="#185FA5")
ax.axvline(best_w, color="crimson", linestyle="--", linewidth=2,
           label=f"Best w={best_w:.2f}")
ax.axvline(0.0, color="#A32D2D", linestyle=":", linewidth=1.5, alpha=0.7, label="Pure log")
ax.axvline(1.0, color="#1D9E75", linestyle=":", linewidth=1.5, alpha=0.7, label="Pure rank")
ax.set_xlabel("Blend weight w  (0=pure log, 1=pure rank)")
ax.set_ylabel("Avg MAE ($k)")
ax.set_title("Blend weight tuning curve", fontweight="bold"); ax.legend(fontsize=8)

# ── Final 3-way MAE comparison ────────────────────────────────────────────────
ax = axes[1]
x = np.arange(len(STORES)); w_b = 0.22
for j, (model_key, lbl_m, alpha, hatch) in enumerate([
    ("rank", "Rank",   0.7, ""),
    ("log",  "Log",    0.85, "//"),
]):
    vals = [results[s][model_key]["mae"]/1000 for s in STORES]
    ax.bar(x+(j-1)*w_b, vals, w_b, label=lbl_m, color=COLORS, alpha=alpha,
           edgecolor="white", hatch=hatch)
# Hybrid
h_vals = [hybrid_results[s]["mae"]/1000 for s in STORES]
ax.bar(x+w_b, h_vals, w_b, label=f"Hybrid (w={best_w:.2f})",
       color=COLORS, edgecolor=[c for c in COLORS], linewidth=2, alpha=1.0)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("MAE ($k)"); ax.set_title("Rank vs Log vs Hybrid MAE", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 12. Complete results table and recommendation

In [ ]:
rows = []
for ts, lbl in zip(STORES, LABELS):
    rank_mae = results[ts]["rank"]["mae"]/1000
    log_mae  = results[ts]["log"]["mae"]/1000
    hyb_mae  = hybrid_results[ts]["mae"]/1000
    best = min([("Rank",rank_mae),("Log",log_mae),("Hybrid",hyb_mae)], key=lambda x:x[1])
    rows.append({
        "Store":       lbl,
        "Rank MAE":    f"${rank_mae:.2f}k",
        "Log MAE":     f"${log_mae:.2f}k",
        "Hybrid MAE":  f"${hyb_mae:.2f}k",
        "Rank R2":     f"{results[ts]['rank']['r2']:.4f}",
        "Log R2":      f"{results[ts]['log']['r2']:.4f}",
        "Rank Sp.":    f"{results[ts]['rank']['spearman']:.4f}",
        "Log Sp.":     f"{results[ts]['log']['spearman']:.4f}",
        "Winner":      best[0],
    })

print(pd.DataFrame(rows).set_index("Store").to_string())

print("\nAverage across all 5 stores:")
for model_key, mlbl in [("rank","Rank"),("log","Log")]:
    avg_mae = np.mean([results[s][model_key]["mae"] for s in STORES])/1000
    avg_r2  = np.mean([results[s][model_key]["r2"]  for s in STORES])
    avg_sp  = np.mean([results[s][model_key]["spearman"] for s in STORES])
    print(f"  {mlbl}: MAE=${avg_mae:.2f}k  R2={avg_r2:.4f}  Spearman={avg_sp:.4f}")
avg_hyb = np.mean([hybrid_results[s]["mae"] for s in STORES])/1000
print(f"  Hybrid: MAE=${avg_hyb:.2f}k")

print("\nRecommendation:")
print("  Use rank model when you can estimate the target income range reliably.")
print("  Use log model when the log-linear assumption holds across domains.")
print("  Use hybrid when uncertain -- it is robust to both failure modes.")
